# Capítulo 4 — Backpropagation e MLP

Códigos do livro *Backpropagation e Redes Neurais* (Araken de Medeiros Santos).

> **Aviso:** este notebook foi gerado a partir dos scripts do livro e **não foi executado como notebook**; as saídas reportadas no texto vêm da execução dos scripts `.py` correspondentes. Execute as células na ordem, a partir desta pasta, com o ambiente de `requirements.txt` (ver Apêndice E). Valores podem diferir ligeiramente em outras plataformas.

In [ ]:
import os, sys
RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(RAIZ, 'codigo'))
os.environ.setdefault('FIGS', os.path.join(RAIZ, 'figs'))
print('rnbook em', os.path.join(RAIZ, 'codigo'))

## Script `cap04/cap04a.py`

In [ ]:
"""Cap. 4 (parte A): exemplo numérico 2-2-1, verificação de gradientes, escalar x vetorizado, XOR, gradientes que desaparecem."""
import sys, os, time
import numpy as np
from rnbook import MLP, grad_numerico, erro_relativo
from rnbook.backprop_escalar import backprop_escalar
from rnbook.ativacoes import sigmoide, d_sigmoide
from rnbook.estilo import plt, COR, salvar
from rnbook.desenho import caixa, seta, diagrama_rede

In [ ]:
# ------------------------------------------------ exemplo 2-2-1 (sigmoide em tudo), uma amostra
W1 = np.array([[0.1, 0.3], [-0.2, 0.4]]); b1 = np.array([0.1, -0.1]); W2 = np.array([[0.5, -0.6]]); b2 = np.array([0.2])
x = np.array([1.0, 0.5]); t = np.array([1.0])
z1 = W1 @ x + b1; a1 = sigmoide(z1); z2 = W2 @ a1 + b2; y = sigmoide(z2); r = y - t; E = 0.5 * float(r @ r)
d2 = r * y * (1 - y); gW2 = np.outer(d2, a1); gb2 = d2
d1 = (W2.T @ d2) * a1 * (1 - a1); gW1 = np.outer(d1, x); gb1 = d1
np.set_printoptions(precision=5, suppress=True)
print("z1", z1, "a1", a1, "z2", z2, "y", y, "r", r, "E %.5f" % E)
print("delta2", d2, "gW2", gW2, "gb2", gb2); print("delta1", d1, "gW1\n", gW1, "gb1", gb1)
rede = MLP([2, 2, 1], ["sigmoide", "sigmoide"], semente=0)
theta = np.concatenate([W1.ravel(), b1, W2.ravel(), b2]); rede.theta = theta.copy()
gv = rede.grad(theta, x[None, :], t[None, :]); gesp = np.concatenate([gW1.ravel(), gb1, gW2.ravel(), gb2])
f = lambda th: rede.perda(th, x[None, :], t[None, :])
gn = grad_numerico(f, theta)
print("à mão x vetorizado: %.2e | vetorizado x numérico: %.2e (erro relativo)" % (np.abs(gv - gesp).max(), erro_relativo(gv, gn)))
E_, dW, db = backprop_escalar([W1, W2], [b1, b2], ["sigmoide", "sigmoide"], x, t)
print("escalar: E=%.5f; dif. máx. vs vetorizado %.2e" % (E_, np.abs(np.concatenate([dW[0].ravel(), db[0], dW[1].ravel(), db[1]]) - gv).max()))
# passo de atualização com eta=0.5
eta = 0.5; th2 = theta - eta * gv; print("E após 1 passo (eta=0.5): %.5f (antes %.5f)" % (rede.perda(th2, x[None, :], t[None, :]), E))

In [ ]:
# ------------------------------------------------ verificação por diferenças finitas: efeito de h
rng = np.random.default_rng(3); X = rng.normal(size=(8, 3)); T = rng.normal(size=(8, 2))
rd = MLP([3, 5, 2], ["tanh", "identidade"], semente=2, init="pequena")
ga = rd.grad(rd.theta, X, T); fe = lambda th: rd.perda(th, X, T)
hs = 10.0 ** -np.arange(1, 14); errs = [erro_relativo(ga, grad_numerico(fe, rd.theta, h)) for h in hs]
print("gradcheck: erro relativo por h:", {f"{h:.0e}": f"{e:.1e}" for h, e in zip(hs, errs)})
print("melhor h:", "%.0e" % hs[int(np.argmin(errs))], "erro %.1e" % min(errs))
# ReLU em ponto de não diferenciabilidade
rr = MLP([1, 1], ["relu"], semente=0); rr.theta = np.array([1.0, 0.0]); Xr = np.array([[0.0]]); Tr = np.array([[1.0]])
print("ReLU em z=0: grad analítico", rr.grad(rr.theta, Xr, Tr), "| numérico", grad_numerico(lambda th: rr.perda(th, Xr, Tr), rr.theta))
fig, ax = plt.subplots(figsize=(5.8, 3.6)); ax.loglog(hs, errs, "o-", color=COR["ind"]); ax.set_xlabel("passo $h$ da diferença central"); ax.set_ylabel("erro relativo do gradiente")
ax.axhline(1e-7, color=COR["grn"], ls="--", lw=1); ax.text(1e-13, 2e-7, "limite usual de aceitação ($10^{-7}$)", color=COR["grn"], fontsize=8); ax.set_title("Verificação de gradiente: escolha de $h$"); salvar(fig, "f4_gradcheck")

In [ ]:
# ------------------------------------------------ escalar x vetorizado: tempo
for N, camadas in [(200, [2, 20, 1])]:
    rr_ = MLP(camadas, ["tanh", "identidade"], semente=1); Xs = np.random.default_rng(0).normal(size=(N, 2)); Ts = np.random.default_rng(1).normal(size=(N, 1))
    Wl = [w.copy() for w, _ in rr_.desempacotar(rr_.theta)]; bl = [b.copy() for _, b in rr_.desempacotar(rr_.theta)]
    t0 = time.perf_counter()
    for _ in range(3): g_esc = sum(np.concatenate([a.ravel() for pair in zip(*backprop_escalar(Wl, bl, ["tanh", "identidade"], Xs[i], Ts[i])[1:]) for a in pair]) for i in range(N)) if False else None
    # soma dos gradientes por amostra (escalar), 3 repetições
    t0 = time.perf_counter()
    for _ in range(3):
        tot = np.zeros(rr_.n_params)
        for i in range(N):
            E_i, dWi, dbi = backprop_escalar(Wl, bl, ["tanh", "identidade"], Xs[i], Ts[i])
            tot += np.concatenate([np.concatenate([dWi[l].ravel(), dbi[l]]) for l in range(2)])
    t_esc = (time.perf_counter() - t0) / 3
    t0 = time.perf_counter()
    for _ in range(300): gv2 = rr_.grad(rr_.theta, Xs, Ts)
    t_vet = (time.perf_counter() - t0) / 300
    print("N=%d, rede %s: escalar %.4f s | vetorizado %.6f s | razão %.0fx | diferença máx. %.2e" % (N, camadas, t_esc, t_vet, t_esc / t_vet, np.abs(tot - gv2).max()))

In [ ]:
# ------------------------------------------------ gradientes que desaparecem: produto de derivadas
L = np.arange(1, 21); print("limite superior do produto das derivadas (sigmoide): 0.25^L para L=5,10,20:", 0.25 ** 5, 0.25 ** 10, 0.25 ** 20)
rngv = np.random.default_rng(0); normas = {}
for nome, ativ in [("sigmoide", "sigmoide"), ("tanh", "tanh"), ("relu", "relu")]:
    r_ = MLP([4] + [20] * 10 + [1], [ativ] * 10 + ["identidade"], semente=5, init="he" if ativ == "relu" else "xavier")
    Xv = rngv.normal(size=(64, 4)); Tv = rngv.normal(size=(64, 1)); g = r_.grad(r_.theta, Xv, Tv)
    sl = r_.fatias(); normas[nome] = [np.linalg.norm(g[sl[l][0][0]:sl[l][0][1]]) for l in range(r_.L)]
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for k, (nome, v) in enumerate(normas.items()): ax.semilogy(range(1, 12), v, "o-", label=nome)
ax.set_xlabel("camada $l$ (1 = mais próxima da entrada)"); ax.set_ylabel("$\\|\\partial E/\\partial W^{(l)}\\|$"); ax.legend(); ax.set_title("Rede com 10 camadas ocultas: norma do gradiente por camada")
salvar(fig, "f4_gradiente_camadas")
print("norma do gradiente na camada 1 / camada 11 :", {k: "%.2e / %.2e" % (v[0], v[-1]) for k, v in normas.items()})

## Script `cap04/cap04b.py`

In [ ]:
"""Cap. 4 (parte B): XOR com backpropagation, aproximação universal, padrão a padrão x minilotes x lote."""
import sys, os, time
import numpy as np
from rnbook import MLP
from rnbook import otimizadores as O
from rnbook.estilo import plt, COR, salvar

In [ ]:
# ---------------------------------------------------------------- XOR
P4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float); T4 = np.array([[0], [1], [1], [0]], float)
def treina_xor(H, semente, eta=1.0, epocas=6000):
    r = MLP([2, H, 1], ["sigmoide", "sigmoide"], semente=semente, init="pequena")
    res = O.gradiente_descendente(r.theta, lambda th: r.perda(th, P4, T4), lambda th: r.grad(th, P4, T4), taxa=eta, max_iter=epocas)
    return r, res
tab = {}
for H in (2, 3, 4):
    ok = 0; ep = []
    for s in range(30):
        r, res = treina_xor(H, s); acertos = ((r.prever(P4, res.theta) > 0.5).astype(float) == T4).all()
        ok += int(acertos); ep.append(res.historico["perda"][-1])
    tab[H] = (ok, np.median(ep)); print("XOR, H=%d: %d/30 sementes resolvem em 6000 épocas (eta=1,0); perda final mediana %.4f" % (H, ok, np.median(ep)))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5), gridspec_kw=dict(width_ratios=[1.15, 1, 1]))
bom = mau = None
for s in range(30):
    r, res = treina_xor(2, s); acert = ((r.prever(P4, res.theta) > 0.5).astype(float) == T4).all()
    cor = COR["grn"] if acert else COR["red"]
    ax[0].semilogy(res.historico["perda"], color=cor, alpha=0.5, lw=1)
    if acert and bom is None: bom = (r, res, s)
    if not acert and mau is None: mau = (r, res, s)
ax[0].set_xlabel("época"); ax[0].set_ylabel("perda $E$ (log)"); ax[0].set_title("XOR com rede 2–2–1: 30 sementes\n(verde: aprende; vermelho: preso)")
gx, gy = np.meshgrid(np.linspace(-0.4, 1.4, 120), np.linspace(-0.4, 1.4, 120)); G = np.c_[gx.ravel(), gy.ravel()]
for a, (nm, tup) in zip(ax[1:], [("aprendeu (semente %d)", bom), ("preso (semente %d)", mau)]):
    if tup is None: a.axis("off"); continue
    r, res, s = tup; Z = r.prever(G, res.theta).reshape(gx.shape)
    a.contourf(gx, gy, Z, levels=np.linspace(0, 1, 11), cmap="RdYlGn", alpha=0.6); a.contour(gx, gy, Z, levels=[0.5], colors="k")
    for p_, t_ in zip(P4, T4[:, 0]): a.scatter(*p_, s=120, color=COR["grn"] if t_ else COR["red"], edgecolor="white", zorder=3)
    a.set_title(nm % s + "\nE=%.4f" % res.historico["perda"][-1]); a.set_aspect("equal"); a.set_xlabel("$x_1$")
salvar(fig, "f4_xor")
if mau: print("exemplo preso: semente", mau[2], "perda final %.4f" % mau[1].historico["perda"][-1], "saídas", np.round(mau[0].prever(P4, mau[1].theta).ravel(), 3))
if bom: print("exemplo bom: semente", bom[2], "épocas até E<0.01:", int(np.argmax(np.array(bom[1].historico["perda"]) < 0.01)), "saídas", np.round(bom[0].prever(P4, bom[1].theta).ravel(), 3))

In [ ]:
# ---------------------------------------------------------------- aproximação universal (ilustração; não é demonstração)
xx = np.linspace(0, 1, 200)[:, None]; alvo = lambda x: np.sin(2 * np.pi * x) + 0.5 * np.sin(6 * np.pi * x); tt = alvo(xx)
xt = np.linspace(0, 1, 1000)[:, None]
resultados = {}
fig, axs = plt.subplots(2, 3, figsize=(11, 5.2))
for ax_, H in zip(axs.ravel(), (1, 2, 4, 8, 16, 32)):
    melhor = None; rm = []
    for s in range(6):
        r = MLP([1, H, 1], ["tanh", "identidade"], semente=s, init="xavier")
        res = O.levenberg_marquardt(r.theta, lambda th: r.residuos(th, xx, tt), lambda th: r.jacobiana(th, xx), max_iter=200)
        e = np.sqrt(np.mean((r.prever(xx, res.theta) - tt) ** 2)); rm.append(e)
        if melhor is None or e < melhor[0]: melhor = (e, r, res.theta, s)
    e, r, th, s = melhor; erro_max = np.max(np.abs(r.prever(xt, th) - alvo(xt)))
    resultados[H] = (e, erro_max, min(rm), max(rm), r.n_params)
    ax_.plot(xt, alvo(xt), color=COR["gry"], lw=2, label="alvo"); ax_.plot(xt, r.prever(xt, th), color=COR["ind"], lw=1.6, label="rede")
    ax_.set_title(f"H={H} ({r.n_params} parâmetros)\nRMSE={e:.4f}  erro máx.={erro_max:.3f}", fontsize=9); ax_.set_xlabel("x")
axs[0, 0].legend(fontsize=8)
salvar(fig, "f4_aproximacao")
for H, v in resultados.items(): print("H=%2d (P=%3d): melhor RMSE %.5f | erro máx (1000 pontos) %.4f | RMSE entre 6 sementes: [%.4f, %.4f]" % (H, v[4], v[0], v[1], v[2], v[3]))
# treinabilidade: GD simples com H=16
r = MLP([1, 16, 1], ["tanh", "identidade"], semente=0)
res = O.gradiente_descendente(r.theta, lambda th: r.perda(th, xx, tt) / 200, lambda th: r.grad(th, xx, tt) / 200, taxa=0.1, max_iter=20000)
print("H=16 com gradiente descendente simples (eta=0.1, 20000 passos): RMSE = %.4f (LM: %.5f)" % (np.sqrt(np.mean((r.prever(xx, res.theta) - tt) ** 2)), resultados[16][0]))

In [ ]:
# ---------------------------------------------------------------- padrão a padrão x minilote x lote
rng = np.random.default_rng(11); N = 200; X = rng.uniform(0, 1, (N, 1)); T = np.sin(2 * np.pi * X) + rng.normal(0, 0.15, (N, 1))
grade = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
modos = [("padrão a padrão (1)", 1), ("minilote (16)", 16), ("lote (200)", N)]
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.6)); linhas = {}; cfg = []
for nome, bs in modos:
    melhor = None
    for eta in grade:
        r = MLP([1, 10, 1], ["tanh", "identidade"], semente=3)
        res = O.gd_estocastico(r.theta, lambda th, idx: r.grad(th, X[idx], T[idx]), N, lambda th: r.perda(th, X, T) / N, taxa=eta, tamanho_lote=bs, epocas=300, semente=0)
        h = np.array(res.historico["perda"]); fin = h[-1] if np.isfinite(h[-1]) else np.inf
        if melhor is None or fin < melhor[0]: melhor = (fin, eta, h, res.tempo)
    fin, eta, h, dt = melhor; cfg.append((nome, bs, eta)); linhas[nome] = (h, dt)
    ax[0].semilogy(h, label=f"{nome}, $\\eta$={eta}")
    print("%-22s melhor eta=%.3f (grade %s): perda(época 10)=%.4f (100)=%.4f (300)=%.4f | tempo 300 épocas %.2f s | atualizações por época: %d" % (nome, eta, grade, h[9], h[99], h[-1], dt, int(np.ceil(N / bs))))
print("piso teórico da perda média E/N com ruído sigma=0.15: %.5f" % (0.5 * 0.15 ** 2))
ax[0].axhline(0.5 * 0.15 ** 2, color=COR["gry"], ls=":", lw=1); ax[0].text(120, 0.0125, "piso do ruído", color=COR["gry"], fontsize=8)
ax[0].set_xlabel("época"); ax[0].set_ylabel("perda média $E/N$"); ax[0].legend(fontsize=8); ax[0].set_title("Convergência por época (melhor $\\eta$ de cada modo)")
ax[1].bar([n for n, _, _ in cfg], [linhas[n][1] for n, _, _ in cfg], color=[COR["red"], COR["amb"], COR["grn"]]); ax[1].set_ylabel("tempo para 300 épocas (s)"); ax[1].set_title("Custo por época (laço em Python)"); ax[1].tick_params(axis="x", labelsize=8)
salvar(fig, "f4_lotes")
# ruído do gradiente: dispersão do gradiente do minilote em torno do gradiente de lote
r = MLP([1, 10, 1], ["tanh", "identidade"], semente=3); gfull = r.grad(r.theta, X, T) / N; rng2 = np.random.default_rng(0)
for bs in (1, 16, 200):
    erros = []
    for _ in range(300):
        idx = rng2.choice(N, bs, replace=False); erros.append(np.linalg.norm(r.grad(r.theta, X[idx], T[idx]) / bs - gfull))
    print("lote %3d: ||g_lote - g_completo|| média %.4f (||g_completo|| = %.4f)" % (bs, np.mean(erros), np.linalg.norm(gfull)))
print("OK cap04b")

## Script `cap04/cap04c.py`

In [ ]:
"""Cap. 4 (parte C): correspondência com o scikit-learn, divisão da amostra, vazamento de informação."""
import sys, os, warnings
warnings.filterwarnings("ignore")
import numpy as np
from rnbook import MLP
from rnbook.estilo import plt, COR, salvar
from rnbook.desenho import caixa, seta
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [ ]:
# ---------------------------------------------------- 1) o nosso MLP reproduz o do scikit-learn?
rng = np.random.default_rng(0); X = rng.uniform(0, 1, (120, 1)); t = (np.sin(2 * np.pi * X) + rng.normal(0, 0.15, X.shape)).ravel()
sk = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", alpha=0.0, max_iter=5000, tol=1e-12, random_state=0).fit(X, t)
W1, W2 = sk.coefs_; b1, b2 = sk.intercepts_
print("formas sklearn: coefs_", [w.shape for w in sk.coefs_], "intercepts_", [b.shape for b in sk.intercepts_], "| out_activation_", sk.out_activation_)
rede = MLP([1, 8, 1], ["tanh", "identidade"]); theta = np.concatenate([W1.T.ravel(), b1, W2.T.ravel(), b2]); rede.theta = theta
print("max |predição própria - sklearn| = %.2e" % np.abs(rede.prever(X, theta).ravel() - sk.predict(X)).max())
Tm = t[:, None]; E_sse = rede.perda(theta, X, Tm, "sse")
print("loss_ do sklearn = %.8f | E_sse/(N*m) = %.8f | E_sse/N = %.8f" % (sk.loss_, E_sse / (len(t) * 1), E_sse / len(t)))
g = rede.grad(theta, X, Tm, "sse"); print("norma do nosso gradiente (SSE) no ponto final do lbfgs do sklearn: %.2e | norma do vetor theta: %.2f" % (np.linalg.norm(g), np.linalg.norm(theta)))
# com alpha > 0
sk2 = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", alpha=0.5, max_iter=5000, tol=1e-12, random_state=0).fit(X, t)
Wa, Wb = sk2.coefs_; th2 = np.concatenate([Wa.T.ravel(), sk2.intercepts_[0], Wb.T.ravel(), sk2.intercepts_[1]]); rede.theta = th2
pen = 0.5 * 0.5 * (np.sum(Wa ** 2) + np.sum(Wb ** 2)) / len(t); print("com alpha=0.5: loss_=%.8f | E_sse/N + penal. = %.8f (penalização só nos pesos, não nos vieses; dividida por N)" % (sk2.loss_, rede.perda(th2, X, Tm, "sse") / len(t) + pen))
# inicialização do sklearn: limite Glorot
print("limite de inicialização (Glorot uniforme) camada 1 (fan_in=1, fan_out=8): tanh sqrt(6/9)=%.4f ; logistic sqrt(2/9)=%.4f" % (np.sqrt(6 / 9), np.sqrt(2 / 9)))

In [ ]:
# ---------------------------------------------------- 2) MLPClassifier: saída e perda
from sklearn.datasets import load_iris, make_moons
Xi, yi = load_iris(return_X_y=True); Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
cl = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(5,), max_iter=2000, random_state=0)).fit(Xtr, ytr)
c = cl[-1]; print("MLPClassifier: out_activation_ =", c.out_activation_, "| classes_", c.classes_, "| n_layers_", c.n_layers_, "| n_iter_", c.n_iter_, "| loss_ %.4f" % c.loss_)
Xb, yb = make_moons(300, noise=0.25, random_state=0); cb = MLPClassifier(hidden_layer_sizes=(5,), max_iter=3000, random_state=0).fit(Xb, yb); print("binário (moons): out_activation_ =", cb.out_activation_, "| coefs_ último:", cb.coefs_[-1].shape)

In [ ]:
# ---------------------------------------------------- 3) divisão da amostra e curva de validação (nº de neurônios)
rng = np.random.default_rng(5); N = 150; Xs = rng.uniform(0, 1, (N, 1)); ts = (np.sin(2 * np.pi * Xs) + rng.normal(0, 0.2, Xs.shape)).ravel()
Xtmp, Xte2, ttmp, tte2 = train_test_split(Xs, ts, test_size=0.2, random_state=0); Xtr2, Xva2, ttr2, tva2 = train_test_split(Xtmp, ttmp, test_size=0.25, random_state=0)
print("tamanhos treino/validação/teste:", len(ttr2), len(tva2), len(tte2))
Hs = [1, 2, 3, 5, 8, 12, 20, 40, 80]; etr, eva = [], []
for H in Hs:
    tr_, va_ = [], []
    for s in range(4):
        m = MLPRegressor(hidden_layer_sizes=(H,), activation="tanh", solver="lbfgs", alpha=0.0, max_iter=3000, random_state=s).fit(Xtr2, ttr2)
        tr_.append(np.sqrt(np.mean((m.predict(Xtr2) - ttr2) ** 2))); va_.append(np.sqrt(np.mean((m.predict(Xva2) - tva2) ** 2)))
    etr.append(np.mean(tr_)); eva.append(np.mean(va_)); print("H=%3d: RMSE treino %.4f | validação %.4f" % (H, etr[-1], eva[-1]))
Hbest = Hs[int(np.argmin(eva))]; print("H escolhido pela validação:", Hbest)
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.7))
# esquema da divisão
a = ax[0]; a.set_xlim(0, 10); a.set_ylim(0, 4); a.axis("off"); a.grid(False)
caixa(a, 0.2, 2.6, 5.6, 0.8, "Treino (60%): ajusta $\\theta$", cor=COR["ind"], fc="#EEF2FF", fs=9)
caixa(a, 5.9, 2.6, 2.0, 0.8, "Validação (20%)", cor=COR["amb"], fc="#FFFBEB", fs=9); caixa(a, 8.0, 2.6, 1.8, 0.8, "Teste (20%)", cor=COR["red"], fc="#FEF2F2", fs=9)
a.text(3.0, 2.2, "gradientes e atualizações", ha="center", fontsize=8.5, color=COR["ind"]); a.text(6.9, 2.2, "escolhas: $H$, $\\eta$,\nparada antecipada", ha="center", fontsize=8.5, color=COR["amb"]); a.text(8.9, 2.2, "uma única vez,\nno final", ha="center", fontsize=8.5, color=COR["red"])
a.text(5, 1.0, "Pré-processamento (média, desvio): ajustado SOMENTE com o treino\ne depois aplicado à validação e ao teste", ha="center", fontsize=9.5, bbox=dict(fc="white", ec=COR["grn"], boxstyle="round"))
a.set_title("Divisão da amostra")
ax[1].semilogx(Hs, etr, "o-", color=COR["ind"], label="treino"); ax[1].semilogx(Hs, eva, "s-", color=COR["amb"], label="validação"); ax[1].axhline(0.2, color=COR["gry"], ls=":", lw=1); ax[1].text(1.2, 0.205, "ruído verdadeiro (0,20)", fontsize=8, color=COR["gry"])
ax[1].scatter([Hbest], [min(eva)], s=120, facecolors="none", edgecolors=COR["red"], zorder=5); ax[1].set_xlabel("neurônios ocultos $H$"); ax[1].set_ylabel("RMSE"); ax[1].legend(); ax[1].set_title("Treino × validação em função de $H$")
salvar(fig, "f4_divisao")
mfin = MLPRegressor(hidden_layer_sizes=(Hbest,), activation="tanh", solver="lbfgs", alpha=0.0, max_iter=3000, random_state=0).fit(np.vstack([Xtr2, Xva2]), np.concatenate([ttr2, tva2]))
print("teste (usado uma vez): RMSE %.4f, R2 %.4f" % (np.sqrt(np.mean((mfin.predict(Xte2) - tte2) ** 2)), r2_score(tte2, mfin.predict(Xte2))))

In [ ]:
# ---------------------------------------------------- 4) vazamento: seleção de atributos fora do pipeline (rótulos aleatórios)
rng = np.random.default_rng(0); Xr = rng.normal(size=(60, 2000)); yr = rng.integers(0, 2, 60)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
Xsel = SelectKBest(f_classif, k=20).fit_transform(Xr, yr)                      # ERRADO: usa todos os rótulos
acc_vaz = cross_val_score(LogisticRegression(max_iter=2000), Xsel, yr, cv=cv).mean()
pipe = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000))
acc_ok = cross_val_score(pipe, Xr, yr, cv=cv).mean()
print("rótulos ALEATÓRIOS (60 amostras, 2000 atributos): validação cruzada com seleção FORA do pipeline = %.3f | DENTRO do pipeline = %.3f (esperado ~0,5)" % (acc_vaz, acc_ok))
print("OK cap04c")

## Script `cap04/cap04d.py`

In [ ]:
"""Cap. 4 (parte D): figuras conceituais (ativações, grafo 2-2-1, fluxo) e moons: nosso MLP x MLPClassifier."""
import sys, os, warnings
warnings.filterwarnings("ignore")
import numpy as np
from rnbook import MLP
from rnbook import otimizadores as O
from rnbook.ativacoes import sigmoide, softmax
from rnbook.estilo import plt, COR, salvar
from rnbook.desenho import caixa, seta
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
# ---------------- f4_ativacoes
z = np.linspace(-5, 5, 400)
fig, ax = plt.subplots(1, 3, figsize=(11.5, 3.3))
for nm, y_, c in [("identidade", z, COR["gry"]), ("sigmoide", sigmoide(z), COR["ind"]), ("tanh", np.tanh(z), COR["grn"]), ("ReLU", np.maximum(0, z), COR["red"]), ("leaky ReLU ($0{,}1$)", np.where(z > 0, z, 0.1 * z), COR["amb"])]:
    ax[0].plot(z, y_, label=nm, color=c, lw=1.8)
ax[0].set_ylim(-2, 3); ax[0].legend(fontsize=7.5, loc="upper left"); ax[0].set_title("Funções de ativação $\\phi(z)$"); ax[0].set_xlabel("z")
zz = np.array([[2.0, 1.0, 0.1]]); sm = softmax(zz)[0]
ax[1].bar(["$z_1$", "$z_2$", "$z_3$"], zz[0], color=COR["gry"], alpha=0.5, label="entrada $z$"); ax[1].bar(["$z_1$", "$z_2$", "$z_3$"], sm, color=COR["ind"], width=0.4, label="saída softmax"); ax[1].legend(fontsize=8)
for i, v in enumerate(sm): ax[1].text(i, v + 0.05, f"{v:.3f}", ha="center", fontsize=9, color=COR["indD"])
ax[1].set_title("Softmax de $z=(2;\\,1;\\,0{,}1)$")
ds = np.linspace(-6, 6, 300); ax[2].plot(ds, sigmoide(ds) * (1 - sigmoide(ds)), color=COR["ind"], label="$\\sigma'(z)$"); ax[2].plot(ds, 1 - np.tanh(ds) ** 2, color=COR["grn"], label="$\\tanh'(z)$"); ax[2].plot(ds, (ds > 0) * 1.0, color=COR["red"], label="ReLU$'(z)$")
ax[2].fill_between(ds, 0, 0.05, where=(abs(ds) > 4), color=COR["amb"], alpha=0.3); ax[2].text(4.6, 0.3, "saturação:\nderivada $\\approx0$", fontsize=8, ha="center", color=COR["amb"]); ax[2].legend(fontsize=8, loc="upper left"); ax[2].set_title("Derivadas das ativações"); ax[2].set_xlabel("z")
salvar(fig, "f4_ativacoes")
print("softmax(2,1,0.1) =", np.round(sm, 4), "soma", sm.sum())

In [ ]:
# ---------------- f4_grafo: grafo computacional 2-2-1 com valores
W1 = np.array([[0.1, 0.3], [-0.2, 0.4]]); b1 = np.array([0.1, -0.1]); W2 = np.array([[0.5, -0.6]]); b2 = np.array([0.2]); x = np.array([1.0, 0.5]); t = 1.0
z1 = W1 @ x + b1; a1 = sigmoide(z1); z2 = W2 @ a1 + b2; y = sigmoide(z2)[0]; r = y - t; d2 = r * y * (1 - y); d1 = (W2.T @ np.array([d2])) * a1 * (1 - a1)
fig, ax = plt.subplots(figsize=(10.4, 4.6)); ax.set_xlim(0, 10.4); ax.set_ylim(0, 4.6); ax.axis("off"); ax.grid(False)
pos = {"x1": (0.7, 3.2), "x2": (0.7, 1.4), "h1": (3.6, 3.2), "h2": (3.6, 1.4), "o": (6.5, 2.3)}
for k, (px, py) in pos.items():
    col = COR["cya"] if k.startswith("x") else COR["ind"] if k.startswith("h") else COR["amb"]
    ax.add_patch(plt.Circle((px, py), 0.45, fc=col, ec="white", lw=1.5, zorder=3)); ax.text(px, py, {"x1": "$x_1$", "x2": "$x_2$", "h1": "$a_1^{(1)}$", "h2": "$a_2^{(1)}$", "o": "$y$"}[k], color="white", ha="center", va="center", fontsize=10, zorder=4)
for i, xi in enumerate(("x1", "x2")):
    for j, hj in enumerate(("h1", "h2")):
        seta(ax, (pos[xi][0] + 0.47, pos[xi][1] - (0.0 if i == j else (0.2 if i < j else -0.2))), (pos[hj][0] - 0.47, pos[hj][1] + (0.0 if i == j else (-0.2 if i < j else 0.2))), cor=COR["gry"], lw=1.1)
for hj in ("h1", "h2"): seta(ax, (pos[hj][0] + 0.47, pos[hj][1] + (-0.2 if hj == "h1" else 0.2)), (pos["o"][0] - 0.47, pos["o"][1] + (0.15 if hj == "h1" else -0.15)), cor=COR["gry"], lw=1.1)
ax.text(0.7, 3.95, "$x=1$", ha="center", fontsize=9); ax.text(0.7, 0.7, "$x=0{,}5$", ha="center", fontsize=9)
ax.text(2.15, 4.3, "$W^{(1)}$: linhas $(0{,}1;\\,0{,}3)$ e $(-0{,}2;\\,0{,}4)$", ha="center", fontsize=8.5, color=COR["ind"])
ax.text(3.6, 3.85, f"$z={z1[0]:.2f}$\n$a={a1[0]:.4f}$", ha="center", fontsize=8.5); ax.text(3.6, 0.65, f"$z={z1[1]:.2f}$\n$a={a1[1]:.4f}$", ha="center", fontsize=8.5)
ax.text(6.5, 3.2, f"$z={z2[0]:.4f}$\n$y={y:.4f}$", ha="center", fontsize=8.5); ax.text(6.5, 1.1, f"$t=1$, $r={r:.4f}$\n$E={0.5*r*r:.5f}$", ha="center", fontsize=8.5, color=COR["red"])
ax.text(8.9, 3.6, "retropropagação", ha="center", fontsize=10, color=COR["red"], fontweight="bold")
ax.text(8.9, 2.85, f"$\\delta^{{(2)}}=r\\,y(1-y)=${d2:.5f}", ha="center", fontsize=9, color=COR["red"])
ax.text(8.9, 2.25, f"$\\delta^{{(1)}}_1=${d1[0]:.5f}", ha="center", fontsize=9, color=COR["red"]); ax.text(8.9, 1.75, f"$\\delta^{{(1)}}_2=${d1[1]:.5f}", ha="center", fontsize=9, color=COR["red"])
ax.text(8.9, 1.05, f"$\\partial E/\\partial W^{{(2)}}=(${d2*a1[0]:.5f}$,\\ ${d2*a1[1]:.5f}$)$", ha="center", fontsize=8.5, color=COR["red"])
ax.set_title("Rede 2–2–1 do exemplo numérico (sigmoide em todos os neurônios)")
salvar(fig, "f4_grafo")

In [ ]:
# ---------------- f4_fluxo: fluxograma do treinamento
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.set_xlim(0, 10); ax.set_ylim(0, 3.2); ax.axis("off"); ax.grid(False)
passos = [("1. Passo direto\n$a^{(l)}=\\phi(W a+b)$", COR["cya"]), ("2. Perda\n$E(\\theta)$", COR["amb"]), ("3. Retropropagação\n$\\delta^{(l)}\\to\\nabla E$", COR["red"]), ("4. Otimizador\n$\\theta\\leftarrow\\theta+\\mathbf{d}$", COR["grn"]), ("5. Parada?\nvalidação, $\\|\\nabla E\\|$", COR["vio"])]
for i, (t_, c) in enumerate(passos):
    caixa(ax, 0.15 + i * 1.97, 1.0, 1.75, 1.3, t_, cor=c, fc="white", fs=8.8, bold=True)
    if i < 4: seta(ax, (0.15 + i * 1.97 + 1.77, 1.65), (0.15 + (i + 1) * 1.97 - 0.02, 1.65), cor=COR["gry"])
seta(ax, (9.0, 1.0), (0.9, 1.0), cor=COR["gry"], rad=0.0, ls="--"); ax.text(5, 0.6, "se não parou: repetir (nova época ou novo lote)", ha="center", fontsize=9, color=COR["gry"])
ax.text(5, 2.9, "O backpropagation é apenas a etapa 3; as etapas 4 e 5 definem o algoritmo de treinamento", ha="center", fontsize=10, style="italic")
salvar(fig, "f4_fluxo")

In [ ]:
# ---------------- moons: nosso MLP (SSE, saída sigmoide, RPROP) x MLPClassifier (lbfgs)
X, y = make_moons(400, noise=0.25, random_state=0); Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
esc = StandardScaler().fit(Xtr); Xs, Xt = esc.transform(Xtr), esc.transform(Xte)
rede = MLP([2, 10, 1], ["tanh", "sigmoide"], semente=1)
res = O.rprop(rede.theta, lambda th: rede.perda(th, Xs, ytr[:, None].astype(float)), lambda th: rede.grad(th, Xs, ytr[:, None].astype(float)), variante="irprop-", max_iter=500)
acc = lambda m, Xa, ya: np.mean((m.prever(Xa, res.theta).ravel() > 0.5) == ya)
print("moons nosso MLP 2-10-1 (SSE, iRPROP-, 500 épocas): acurácia treino %.4f | teste %.4f | P=%d" % (acc(rede, Xs, ytr), acc(rede, Xt, yte), rede.n_params))
sk = MLPClassifier(hidden_layer_sizes=(10,), activation="tanh", solver="lbfgs", max_iter=2000, random_state=1).fit(Xs, ytr)
print("moons MLPClassifier (tanh, lbfgs, log_loss): treino %.4f | teste %.4f" % (sk.score(Xs, ytr), sk.score(Xt, yte)))
sk2 = MLPClassifier(hidden_layer_sizes=(10,), activation="relu", solver="adam", max_iter=2000, random_state=1).fit(Xs, ytr)
print("moons MLPClassifier (relu, adam padrão): treino %.4f | teste %.4f | n_iter_ %d" % (sk2.score(Xs, ytr), sk2.score(Xt, yte), sk2.n_iter_))
gx, gy = np.meshgrid(np.linspace(-2.6, 2.8, 250), np.linspace(-2.2, 2.4, 250)); G = np.c_[gx.ravel(), gy.ravel()]
fig, ax = plt.subplots(1, 2, figsize=(10.2, 3.9))
for a, (nm, f_) in zip(ax, [("nosso MLP 2–10–1 (SSE, iRPROP−)", lambda G_: rede.prever(G_, res.theta).ravel()), ("MLPClassifier (tanh, lbfgs, entropia cruzada)", lambda G_: sk.predict_proba(G_)[:, 1])]):
    Z = f_(G).reshape(gx.shape); a.contourf(gx, gy, Z, levels=np.linspace(0, 1, 11), cmap="RdYlGn", alpha=0.6); a.contour(gx, gy, Z, levels=[0.5], colors="k", linewidths=1.5)
    a.scatter(Xs[:, 0], Xs[:, 1], c=np.where(ytr == 1, COR["grn"], COR["red"]), s=14, edgecolor="white", linewidths=0.3); a.set_title(nm, fontsize=9); a.set_xlabel("$x_1$ (padronizado)")
salvar(fig, "f4_moons")
print("OK cap04d")

## Exercícios práticos e projeto: soluções de referência

### 1

Implemente o passo direto e o passo reverso para uma rede 2–2–1 com sigmoide em todos os neurônios e confira com os números do Exemplo 4.2.

1. Escreva `forward(x, t, params)` e `backward(...)` com as Equações BP1 a BP4.
2. Com $\mathbf{x}=(1;0{,}5)$, $t=1$ e os parâmetros do Exemplo 4.2, imprima $E$, $\delta^{(2)}$, $\boldsymbol{\delta}^{(1)}$ e todos os gradientes.
3. Verifique cada gradiente por diferenças centrais ($h=10^{-5}$) e imprima o erro relativo.
4. Execute 5 passos de gradiente descendente com $\eta=0{,}5$ e mostre $E$ a cada passo.

**Pergunta:** o que acontece com o erro relativo se você usar $h=10^{-12}$?

In [ ]:
import numpy as np

sig = lambda z: 1.0 / (1.0 + np.exp(-z))
x = np.array([1.0, 0.5]); t = np.array([1.0])
P = {"W1": np.array([[0.1, 0.3], [-0.2, 0.4]]), "b1": np.array([0.1, -0.1]),
     "W2": np.array([[0.5, -0.6]]), "b2": np.array([0.2])}

def forward(P):
    a1 = sig(P["W1"] @ x + P["b1"]); y = sig(P["W2"] @ a1 + P["b2"])
    return a1, y, 0.5 * np.sum((y - t) ** 2)

def backward(P):
    a1, y, E = forward(P)
    d2 = (y - t) * y * (1 - y)                          # BP1
    d1 = (P["W2"].T @ d2) * a1 * (1 - a1)               # BP2
    return {"W2": np.outer(d2, a1), "b2": d2, "W1": np.outer(d1, x), "b1": d1}, d2, d1   # BP3, BP4

g, d2, d1 = backward(P)
print("E = %.5f | delta2 = %s | delta1 = %s" % (forward(P)[2], np.round(d2, 5), np.round(d1, 5)))
for nome in g: print(nome, np.round(g[nome], 5).tolist())

def erro_rel(P, h):
    pior = 0.0
    for nome in P:
        gn = np.zeros_like(P[nome])
        for idx in np.ndindex(*P[nome].shape):
            Pp = {k: v.copy() for k, v in P.items()}; Pm = {k: v.copy() for k, v in P.items()}
            Pp[nome][idx] += h; Pm[nome][idx] -= h
            gn[idx] = (forward(Pp)[2] - forward(Pm)[2]) / (2 * h)
        pior = max(pior, np.linalg.norm(g[nome] - gn) / (np.linalg.norm(g[nome]) + np.linalg.norm(gn)))
    return pior
print("erro relativo (h=1e-5): %.2e | (h=1e-12): %.2e" % (erro_rel(P, 1e-5), erro_rel(P, 1e-12)))

Q = {k: v.copy() for k, v in P.items()}
for passo in range(5):
    g, _, _ = backward(Q)
    for k in Q: Q[k] -= 0.5 * g[k]
    print("passo %d: E = %.5f" % (passo + 1, forward(Q)[2]))

### 2

Verifique, com o scikit-learn, se uma rede pequena aprende o XOR.

1. Monte os quatro pontos do XOR.
2. Para $H\in\{2,\,4\}$, treine `MLPClassifier(hidden_layer_sizes=(H,), activation="tanh", solver="lbfgs", max_iter=2000)` com 20 sementes (`random_state=0,…,19`) e conte em quantas a acurácia nos quatro pontos é 1,0.
3. Repita com `solver="sgd"`, `learning_rate_init=0.1` e `max_iter=2000` (apenas para $H=2$).

**Pergunta:** o que isso diz sobre a "garantia" do teorema da aproximação universal?

In [ ]:
import warnings
import numpy as np
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float); y = np.array([0, 1, 1, 0])

def taxa(H, solver, **kw):
    ok = 0
    for s in range(20):
        m = MLPClassifier(hidden_layer_sizes=(H,), activation="tanh", solver=solver, max_iter=2000,
                          random_state=s, **kw).fit(X, y)
        ok += int(m.score(X, y) == 1.0)
    return ok

print("H=2, lbfgs:", taxa(2, "lbfgs"), "/ 20")
print("H=4, lbfgs:", taxa(4, "lbfgs"), "/ 20")
print("H=2, sgd (taxa 0,1):", taxa(2, "sgd", learning_rate_init=0.1), "/ 20")

### 3

A base **Diabetes** (Efron et al., 2004; 442 pacientes, 10 atributos padronizados, alvo = progressão da doença) é um problema de regressão pequeno e ruidoso.

1. Separe treino/validação/teste (60/20/20, `random_state=0`).
2. Treine `LinearRegression` (linha de base) e `MLPRegressor(hidden_layer_sizes=(H,), solver="lbfgs", alpha=a, max_iter=5000)` para $H\in\{2,5,10,30\}$ e $a\in\{0;\,1;\,10\}$.
3. Escolha $(H,a)$ pelo $R^2$ de validação e avalie no teste **uma vez**, junto com a linha de base.

**Pergunta:** o MLP trouxe ganho? O que a regularização `alpha` faz?

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
X, y = load_diabetes(return_X_y=True)
X_a, X_te, y_a, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_a, y_a, test_size=0.25, random_state=0)

base = LinearRegression().fit(X_tr, y_tr)
print("linear: R2 validação = %.4f" % base.score(X_va, y_va))
melhor = None
for H in (2, 5, 10, 30):
    for a in (0.0, 1.0, 10.0):
        mod = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(H,), solver="lbfgs", alpha=a,
                                                           max_iter=5000, random_state=0)).fit(X_tr, y_tr)
        r2 = mod.score(X_va, y_va)
        print("H=%2d alpha=%4.1f  R2 treino=%.3f validação=%.3f" % (H, a, mod.score(X_tr, y_tr), r2))
        if melhor is None or r2 > melhor[0]: melhor = (r2, H, a)
r2, H, a = melhor
final = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(H,), solver="lbfgs", alpha=a,
                                                     max_iter=5000, random_state=0)).fit(X_a, y_a)
print("escolhido: H=%d alpha=%.1f | teste: MLP R2=%.4f | linear R2=%.4f"
      % (H, a, final.score(X_te, y_te), LinearRegression().fit(X_a, y_a).score(X_te, y_te)))

### Reconhecendo dígitos: arquitetura, otimizador e generalização

**Objetivo.** Estudar como a arquitetura e o otimizador afetam o desempenho de um MLP em uma base real de imagens.

**Dados.** A base **Digits** do scikit-learn: 1797 imagens $8\times8$ de dígitos manuscritos (subconjunto da base *Optical Recognition of Handwritten Digits*, do repositório da UCI), 64 atributos, 10 classes.

**Tarefas.**
1. Separe 60% treino, 20% validação e 20% teste (estratificado, `random_state=0`). Padronize com estatísticas **do treino**.
2. Para $H\in\{10,\,30,\,100\}$ e `solver` em {`lbfgs`, `adam`}, treine com 3 sementes e registre: acurácia de validação (média ± desvio), tempo de treinamento (s) e número de iterações.
3. Escolha a configuração pela validação, treine-a de novo com treino + validação e avalie no teste **uma única vez**. Mostre a matriz de confusão.
4. Redija 10 a 12 linhas: efeito da largura, diferença entre os otimizadores (qual custo? qual precisão?), e limites do estudo.

**Para ir além (não testado no livro).** Obtenha, no repositório da UCI, o conjunto original *optdigits* (treino e teste separados, 32×32 reduzidos a 8×8) e, no Kaggle, bases de dígitos como o MNIST; compare o desempenho e o tempo de treinamento. Confira licença e formato antes de usar.

In [ ]:
import time
import warnings
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
X, y = load_digits(return_X_y=True)
X_a, X_te, y_a, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_a, y_a, test_size=0.25, random_state=0, stratify=y_a)
print("treino/validação/teste:", len(y_tr), len(y_va), len(y_te))

def construir(H, solver, semente):
    return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(H,), solver=solver,
                                                         max_iter=500, random_state=semente))
res = {}
for solver in ("lbfgs", "adam"):
    for H in (10, 30, 100):
        acc, tempo, its = [], [], []
        for s in range(3):
            t0 = time.perf_counter(); m = construir(H, solver, s).fit(X_tr, y_tr); tempo.append(time.perf_counter() - t0)
            acc.append(m.score(X_va, y_va)); its.append(m[-1].n_iter_)
        res[(solver, H)] = np.mean(acc)
        print("%-6s H=%3d  acc_val = %.4f ± %.4f | tempo = %.2f s | iterações = %d"
              % (solver, H, np.mean(acc), np.std(acc), np.mean(tempo), np.mean(its)))
solver, H = max(res, key=res.get)
final = construir(H, solver, 0).fit(np.vstack([X_tr, X_va]), np.concatenate([y_tr, y_va]))
print("escolhida:", solver, H, "| acurácia no teste (uma vez): %.4f" % final.score(X_te, y_te))
print(confusion_matrix(y_te, final.predict(X_te)))